# RAG

Advanced RAG Techniques!


1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [6]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)

MODEL = "ollama/llama3"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("../knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()

In [7]:
class Result(BaseModel):
    page_content: str
    metadata: dict

In [8]:
class Chunk(BaseModel):
    headline: str = Field(description="Un breve título para este fragmento, normalmente de unas pocas palabras, que es el que más probabilidades tiene de aparecer en una búsqueda.")
    summary: str = Field(description="Unas cuantas frases que resuman el contenido de este fragmento para responder a las preguntas más habituales")
    original_text: str = Field(description="El texto original de este fragmento del documento facilitado, tal y como está, sin modificar en absoluto")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]


1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

In [9]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [10]:
documents = fetch_documents()

Loaded 15 documents


### Step 2 - make the chunks

In [11]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
      Tomas un documento y lo divides en fragmentos superpuestos para una base de conocimientos.
    
      El documento procede de la unidad compartida de una empresa llamada Insurellm.
      El documento es de tipo: {document["type"]}
      El documento se ha obtenido de: {document["source"]}
    
    Un chatbot utilizará estos fragmentos para responder a preguntas sobre la empresa.
    Debes dividir el documento como mejor te parezca, asegurándote de que todo el documento quede incluido en los fragmentos; no te dejes nada fuera.
    Probablemente, este documento debería dividirse en {how_many} fragmentos, pero puedes tener más o menos según sea necesario.
    Debe haber solapamiento entre los fragmentos según sea necesario; normalmente, un solapamiento de alrededor del 25 % o unas 50 palabras, de modo que el mismo texto aparezca en varios fragmentos para obtener los mejores resultados de recuperación.
    
    Para cada fragmento, debes proporcionar un título, un resumen y el texto original del fragmento.
    En conjunto, tus fragmentos deben representar el documento completo con solapamiento.
    
    Aquí está el documento:
    
    {document["text"]}
    
    Responde con los fragmentos.
"""

In [12]:
print(make_prompt(documents[0]))


      Tomas un documento y lo divides en fragmentos superpuestos para una base de conocimientos.

      El documento procede de la unidad compartida de una empresa llamada Insurellm.
      El documento es de tipo: compañia
      El documento se ha obtenido de: ../knowledge-base/compañia/carreras.md

    Un chatbot utilizará estos fragmentos para responder a preguntas sobre la empresa.
    Debes dividir el documento como mejor te parezca, asegurándote de que todo el documento quede incluido en los fragmentos; no te dejes nada fuera.
    Probablemente, este documento debería dividirse en 17 fragmentos, pero puedes tener más o menos según sea necesario.
    Debe haber solapamiento entre los fragmentos según sea necesario; normalmente, un solapamiento de alrededor del 25 % o unas 50 palabras, de modo que el mismo texto aparezca en varios fragmentos para obtener los mejores resultados de recuperación.

    Para cada fragmento, debes proporcionar un título, un resumen y el texto original de

In [13]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [14]:
make_messages(documents[0])

[{'role': 'user',
  'content': '\n      Tomas un documento y lo divides en fragmentos superpuestos para una base de conocimientos.\n\n      El documento procede de la unidad compartida de una empresa llamada Insurellm.\n      El documento es de tipo: compañia\n      El documento se ha obtenido de: ../knowledge-base/compañia/carreras.md\n\n    Un chatbot utilizará estos fragmentos para responder a preguntas sobre la empresa.\n    Debes dividir el documento como mejor te parezca, asegurándote de que todo el documento quede incluido en los fragmentos; no te dejes nada fuera.\n    Probablemente, este documento debería dividirse en 17 fragmentos, pero puedes tener más o menos según sea necesario.\n    Debe haber solapamiento entre los fragmentos según sea necesario; normalmente, un solapamiento de alrededor del 25 % o unas 50 palabras, de modo que el mismo texto aparezca en varios fragmentos para obtener los mejores resultados de recuperación.\n\n    Para cada fragmento, debes proporcionar 

In [15]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [16]:
process_document(documents[0])

[Result(page_content='# Empleo en AgroLLM\n\nEn AgroLLM no solo desarrollamos software: estamos revolucionando el sector agropecuario.\n\n# Empleo en AgroLLM\n\n## ¿Por qué unirte a AgroLLM?\n\nEn AgroLLM no solo desarrollamos software: estamos revolucionando el sector agropecuario. Desde nuestra fundación en 2015, hemos evolucionado de ser una startup de rápido crecimiento a una empresa altamente rentable y eficiente.', metadata={'source': '../knowledge-base/compañia/carreras.md', 'type': 'compañia'}),
 Result(page_content='Oportunidades Actuales\n\nNuestras oportunidades laborales actuales incluyen posiciones en ingeniería y sistemas de IA, datos, análisis y agronomía.\n\n\n## Oportunidades Actuales\n\n### Ingeniería y Sistemas de IA\n\n**Ingeniero/a Senior Full Stack (Django & React)** - Santa Cruz de Tenerife (Híbrido) / Remoto\n...', metadata={'source': '../knowledge-base/compañia/carreras.md', 'type': 'compañia'}),
 Result(page_content='Ingeniería y Sistemas de IA\n\nBuscamos ing

In [17]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [18]:
chunks = create_chunks(documents)

100%|██████████| 15/15 [10:42<00:00, 42.85s/it]


In [19]:
print(len(chunks))

101


### ¡Para optimizar la estrategia de chunking!

En la versión del módulo de Python, utilizo discretamente el «Pool» de multiprocesamiento para ejecutar esto en paralelo,
pero si te aparece un error de límite de velocidad, puedes desactivarlo en el código.

### Por último, paso 3: guardar las representaciones

In [20]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [21]:
create_embeddings(chunks)

Vectorstore created with 101 documents


In [24]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['productos', 'empleados', 'compañia'].index(t)] for t in doc_types]

In [25]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [26]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## RAG avanzado

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [27]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="Orden de relevancia de los fragmentos, de más relevante a menos relevante, según el número de identificación de cada fragmento"
    )

In [28]:
def rerank(question, chunks):
    system_prompt = """
        Eres un sistema de reordenación de documentos.
        Se te proporciona una pregunta y una lista de fragmentos de texto relevantes extraídos de una consulta a una base de conocimientos.
        Los fragmentos se proporcionan en el orden en que se han recuperado; este orden debería estar aproximadamente ordenado por relevancia, pero es posible que puedas mejorarlo.
        Debes clasificar los fragmentos proporcionados por orden de relevancia respecto a la pregunta, colocando el más relevante en primer lugar.
        Responde únicamente con la lista de identificadores de los fragmentos clasificados, nada más. Incluye todos los identificadores de fragmentos que se te hayan proporcionado, reordenados.
    """
    user_prompt = f"El usuario ha formulado la siguiente pregunta:\n\n{question}\n\nOrdena todos los fragmentos de texto por relevancia respecto a la pregunta, de más relevante a menos relevante. Incluye todos los identificadores de fragmento que se te han facilitado, reordenados.\n\n"
    user_prompt += "Aquí están los fragmentos:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Responde únicamente con la lista de los identificadores de fragmentos ordenados por prioridad; nada más."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [32]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [30]:
question = "¿Logró recuperar el 100% de los incentivos fiscales del POSEI asignados?"
chunks = fetch_context_unranked(question)

In [31]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

Integración con...
Integración con...
Características...
PoseiLLM (Ayuda...
Características...
Características...
Características...
Ventajas



Ven...
Annual Performa...
Evaluación de e...


In [33]:
reranked = rerank(question, chunks)

[9, 2, 5]


In [26]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

Insurellm Caree...
Other HR Notes ...
Annual Performa...
Career Progress...
Annual Performa...
Annual Performa...
Performance His...
Performance and...
IoT Device Inte...
Career Progress...


In [34]:
question = "¿Quien dispone de un tanque de 500 pipass?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

In [35]:
reranked = rerank(question, chunks)

[2, 9, 5, 1, 11, 17, 3]


In [36]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [37]:
reranked[0].page_content

'Simulador de eficiencia energética en bombeos\n\nHerramienta analítica que evalúa los costes eléctricos asociados al bombeo de agua desde pozos profundos o redes presurizadas.\n\nHerramienta analítica que evalúa los costes eléctricos asociados al bombeo de agua desde pozos profundos o redes presurizadas. Sugiere los horarios de riego más económicos basándose en las tarifas eléctricas indexadas del mercado español y la disponibilidad de almacenamiento en estanques propios.'

In [38]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [39]:
SYSTEM_PROMPT = """
Eres un asistente experto y amable que representa a la empresa Insurellm.
Estás chateando con un usuario sobre Insurellm.
Tu respuesta se evaluará en cuanto a precisión, relevancia y exhaustividad, así que asegúrate de que se limite a responder a la pregunta y lo haga de forma completa.
Si no sabes la respuesta, dilo.
Para contextualizar, aquí tienes algunos extractos específicos de la base de conocimientos que podrían ser directamente relevantes para la pregunta del usuario
{context}

Teniendo en cuenta este contexto, responde a la pregunta del usuario. Sé preciso, relevante y exhaustivo.
"""

In [40]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [ ]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
        Estás manteniendo una conversación con un usuario y respondiendo a preguntas sobre la empresa AgroTech.
        Estás a punto de buscar información en una base de conocimientos para responder a la pregunta del usuario.

        Este es el historial de tu conversación con el usuario hasta el momento:
        {history}

        Y esta es la pregunta actual del usuario:
        {question}

        Responde únicamente con una única pregunta concisa que vas a utilizar para buscar en la base de conocimientos.
        Debe ser una pregunta MUY breve y específica que tenga más probabilidades de dar resultados. Céntrate en los detalles de la pregunta.
        No menciones el nombre de la empresa, a menos que se trate de una pregunta general sobre la misma.
        IMPORTANTE: Responde ÚNICAMENTE con la consulta de la base de conocimientos, nada más.
    """
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [43]:
rewrite_query("¿Que producto tiene el Módulo de análisis de costes de insumos (fertilizantes y energía)", [])

'¿Qué productos incluyen análisis de costos de fertilizantes y energía en Insurellm?'

In [45]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [48]:
answer_question("¿Quién es Arcadio?", [])

¿Quién es Arcadio en Insurellm?
[2, 9, 10, 1, 3, 4, 5, 7, 12, 13, 14, 15, 17]


('Lo siento, pero no hay información disponible sobre una persona llamada Arcadio en el contexto de Insurellm o AgroLLM. Como asistente experto y amable de la empresa, mi objetivo es brindarte respuestas precisas y relevantes relacionadas con nuestros productos y servicios. Si tienes alguna pregunta más específica o deseas obtener información sobre uno de nuestros productos o soluciones, estaré encantado de ayudarte.',
 [Result(page_content='Empoderar a los Productores del Sector Primario\n\nLa declaración de misión de Insurellm se centra en empoderar a los productores del sector primario con soluciones de software de vanguardia.\n\nEmpoderar a los productores del sector primario, cooperativas y oficinas de extensión agraria con soluciones de software de vanguardia que simplifiquen la burocracia, optimicen los recursos hídricos en zonas áridas y mejoren la toma de decisiones en el campo. Combinando el conocimiento técnico tradicional con la innovación de la Inteligencia Artificial, est